# 01_preprocessing — 원본 CSV를 다루기 좋은 형태로 변환

**목적(Why)**: 원본 GFS/LDAPS 기상 데이터는 `forecast_kst_dtm` 하나당 격자(grid_id)별로 여러 행이 있는 '긴 형태(long format)'입니다. 이 상태로는 시간별 발전량과 바로 조인할 수 없고, 모델 입력으로 쓰기도 불편합니다. 이 노트북은 **구조만 바꾸는 단계**입니다 — 어떤 피처를 쓸지 고르거나 새 변수를 만들지는 않습니다 (그건 `03_features.ipynb`의 몫). SCADA도 10분 단위를 1시간 단위로 맞추고 KPX 그룹별로 합칩니다.

**입력**: `data/train/*.csv`, `data/test/*.csv`, `data/info.xlsx`
**출력**: `data/processed/train_base.parquet`, `data/processed/test_base.parquet`, `data/processed/{gfs,ldaps}_grid_meta.parquet`

**중요한 원칙(leakage-guard)**: train과 test는 반드시 **같은 함수, 같은 로직**으로 처리합니다. 아래 셀들은 함수로 정의한 뒤 train/test에 각각 호출하는 방식입니다.

## 0. 환경 설정

In [1]:
import sys
print(sys.executable)  # venv 안의 파이썬을 쓰고 있는지 확인

d:\공모전\wind_forecast_new\venv\Scripts\python.exe


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 50)
SEED = 42
np.random.seed(SEED)

DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CAPACITY_KWH = {
    "kpx_group_1": 21600,
    "kpx_group_2": 21600,
    "kpx_group_3": 21000,
}
GROUP_COLS = list(CAPACITY_KWH.keys())

## 1. GFS/LDAPS 기상 데이터 — 긴 형태 → 넓은 형태(pivot)

**지금 하는 일**: `forecast_kst_dtm` + `grid_id`로 되어 있는 긴 형태를, `forecast_kst_dtm` 한 행에 격자별 변수를 옆으로 펼친 넓은 형태로 바꿉니다. 예를 들어 GFS는 격자가 9개이므로, 원래 1개였던 `heightAboveGround_10_10u` 컬럼이 `gfs_g1_heightAboveGround_10_10u` ~ `gfs_g9_heightAboveGround_10_10u`처럼 9개 컬럼으로 펼쳐집니다.

**왜 지금 격자를 고르지 않고 다 남겨두나요?**: 어느 격자가 터빈에 가장 가까운지(최근접 격자)는 EDA에서 위경도를 직접 봐야 판단할 수 있는 문제라, 지금 임의로 격자를 줄이면 나중에 되돌리기 어렵습니다. 원칙(2.5절: 근거 없이 변수를 고르지 않는다)에 따라 지금은 정보를 보존하고, 격자 선택은 `03_features.ipynb`에서 EDA 근거를 가지고 결정합니다.

**위경도는 왜 넓은 표에서 빼나요?**: 위경도는 격자마다 고정된 값(시간에 따라 안 변함)이라 매 시간 행에 반복해서 넣을 필요가 없습니다. 대신 격자 지도(위경도)만 따로 작은 표로 저장해서, EDA에서 '격자-터빈 위치 지도'를 그릴 때 씁니다.

In [3]:
def pivot_nwp(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    """forecast_kst_dtm x grid_id 긴 형태 -> forecast_kst_dtm 1행 넓은 형태로 변환."""
    meta_cols = ["forecast_kst_dtm", "data_available_kst_dtm", "grid_id", "latitude", "longitude"]
    value_cols = [c for c in df.columns if c not in meta_cols]

    wide = df.pivot(index="forecast_kst_dtm", columns="grid_id", values=value_cols)
    wide.columns = [f"{source_name}_g{grid_id}_{var}" for var, grid_id in wide.columns]

    # data_available_kst_dtm은 같은 forecast_kst_dtm 안에서 grid_id와 무관하게 동일해야 함 (아래 확인 셀에서 검증)
    avail = df.groupby("forecast_kst_dtm")["data_available_kst_dtm"].first()
    wide = wide.join(avail)

    return wide.reset_index()


def grid_meta(df: pd.DataFrame) -> pd.DataFrame:
    """격자별 위경도 메타 정보만 따로 추출 (EDA 지도용)."""
    return df[["grid_id", "latitude", "longitude"]].drop_duplicates().sort_values("grid_id").reset_index(drop=True)

### 1-1. GFS 적용 (train/test 동일 로직)

In [4]:
gfs_train_raw = pd.read_csv(DATA_DIR / "train" / "gfs_train.csv", encoding="utf-8-sig", parse_dates=["forecast_kst_dtm", "data_available_kst_dtm"])
gfs_test_raw = pd.read_csv(DATA_DIR / "test" / "gfs_test.csv", encoding="utf-8-sig", parse_dates=["forecast_kst_dtm", "data_available_kst_dtm"])

print(gfs_train_raw.shape, gfs_test_raw.shape)
print("train grid_id 개수:", gfs_train_raw["grid_id"].nunique(), "| test grid_id 개수:", gfs_test_raw["grid_id"].nunique())

(236736, 40) (78840, 40)
train grid_id 개수: 9 | test grid_id 개수: 9


In [5]:
gfs_grid_meta = grid_meta(gfs_train_raw)
gfs_train = pivot_nwp(gfs_train_raw, "gfs")
gfs_test = pivot_nwp(gfs_test_raw, "gfs")

print(gfs_train.shape, gfs_test.shape)
gfs_train.head(2)

(26304, 317) (8760, 317)


,forecast_kst_dtm,gfs_g1_heightAboveGround_10_10u,gfs_g2_heightAboveGround_10_10u,gfs_g3_heightAboveGround_10_10u,gfs_g4_heightAboveGround_10_10u,gfs_g5_heightAboveGround_10_10u,gfs_g6_heightAboveGround_10_10u,gfs_g7_heightAboveGround_10_10u,gfs_g8_heightAboveGround_10_10u,gfs_g9_heightAboveGround_10_10u,gfs_g1_heightAboveGround_10_10v,gfs_g2_heightAboveGround_10_10v,gfs_g3_heightAboveGround_10_10v,gfs_g4_heightAboveGround_10_10v,gfs_g5_heightAboveGround_10_10v,gfs_g6_heightAboveGround_10_10v,gfs_g7_heightAboveGround_10_10v,gfs_g8_heightAboveGround_10_10v,gfs_g9_heightAboveGround_10_10v,gfs_g1_heightAboveGround_80_u,gfs_g2_heightAboveGround_80_u,gfs_g3_heightAboveGround_80_u,gfs_g4_heightAboveGround_80_u,gfs_g5_heightAboveGround_80_u,gfs_g6_heightAboveGround_80_u,...,gfs_g4_isobaricInhPa_500_t,gfs_g5_isobaricInhPa_500_t,gfs_g6_isobaricInhPa_500_t,gfs_g7_isobaricInhPa_500_t,gfs_g8_isobaricInhPa_500_t,gfs_g9_isobaricInhPa_500_t,gfs_g1_isobaricInhPa_500_u,gfs_g2_isobaricInhPa_500_u,gfs_g3_isobaricInhPa_500_u,gfs_g4_isobaricInhPa_500_u,gfs_g5_isobaricInhPa_500_u,gfs_g6_isobaricInhPa_500_u,gfs_g7_isobaricInhPa_500_u,gfs_g8_isobaricInhPa_500_u,gfs_g9_isobaricInhPa_500_u,gfs_g1_isobaricInhPa_500_v,gfs_g2_isobaricInhPa_500_v,gfs_g3_isobaricInhPa_500_v,gfs_g4_isobaricInhPa_500_v,gfs_g5_isobaricInhPa_500_v,gfs_g6_isobaricInhPa_500_v,gfs_g7_isobaricInhPa_500_v,gfs_g8_isobaricInhPa_500_v,gfs_g9_isobaricInhPa_500_v,data_available_kst_dtm
0,2022-01-01 01:00:00,1.817466,2.947466,2.817466,1.227466,2.457466,2.247466,0.997466,1.617466,2.827466,1.040068,1.340068,0.800068,0.190068,0.540068,0.490068,-1.499932,-0.099932,0.680068,2.437944,3.747944,2.577944,2.037944,3.107944,2.347944,...,244.00014,244.14014,244.28014,244.71014,244.25014,244.36014,22.805730,23.905731,24.905731,23.605732,24.205730,25.005732,25.305730,24.905731,25.005732,-15.617361,-16.017360,-16.317362,-14.717361,-15.317362,-15.517362,-14.117361,-13.717361,-13.717361,2021-12-31 13:00:00
1,2022-01-01 02:00:00,1.781655,2.861655,2.851655,1.401655,2.391655,2.091655,1.011655,1.651655,2.831655,1.069929,1.289929,0.819929,0.319929,0.629929,0.459929,-1.390071,0.229929,0.709929,2.365549,3.635549,2.575549,2.225549,2.985549,2.105549,...,244.01562,244.10562,244.12563,244.96562,244.67563,244.67563,21.762934,22.762934,23.362934,22.962933,23.262934,23.862934,25.562933,25.062933,25.062933,-14.392309,-14.892309,-15.292310,-13.392309,-14.092310,-14.392309,-13.492310,-12.892309,-13.192309,2021-12-31 13:00:00


**확인할 것**
- `gfs_train.shape[0]`이 26304(train_labels 행 수)와 같은지, `gfs_test.shape[0]`이 8760과 같은지
- 컬럼 수가 `(9개 격자 x 변수 수) + 2(forecast_kst_dtm, data_available_kst_dtm)`와 맞는지
- `gfs_train_raw["grid_id"].nunique()`가 9인지

### 1-2. LDAPS 적용 (train/test 동일 로직, GFS와 같은 함수 재사용)

In [6]:
ldaps_train_raw = pd.read_csv(DATA_DIR / "train" / "ldaps_train.csv", encoding="utf-8-sig", parse_dates=["forecast_kst_dtm", "data_available_kst_dtm"])
ldaps_test_raw = pd.read_csv(DATA_DIR / "test" / "ldaps_test.csv", encoding="utf-8-sig", parse_dates=["forecast_kst_dtm", "data_available_kst_dtm"])

print(ldaps_train_raw.shape, ldaps_test_raw.shape)
print("train grid_id 개수:", ldaps_train_raw["grid_id"].nunique(), "| test grid_id 개수:", ldaps_test_raw["grid_id"].nunique())

(420864, 35) (140160, 35)
train grid_id 개수: 16 | test grid_id 개수: 16


In [7]:
ldaps_grid_meta = grid_meta(ldaps_train_raw)
ldaps_train = pivot_nwp(ldaps_train_raw, "ldaps")
ldaps_test = pivot_nwp(ldaps_test_raw, "ldaps")

print(ldaps_train.shape, ldaps_test.shape)
ldaps_train.head(2)

(26304, 482) (8760, 482)


,forecast_kst_dtm,ldaps_g1_heightAboveGround_10_10u,ldaps_g2_heightAboveGround_10_10u,ldaps_g3_heightAboveGround_10_10u,ldaps_g4_heightAboveGround_10_10u,ldaps_g5_heightAboveGround_10_10u,ldaps_g6_heightAboveGround_10_10u,ldaps_g7_heightAboveGround_10_10u,ldaps_g8_heightAboveGround_10_10u,ldaps_g9_heightAboveGround_10_10u,ldaps_g10_heightAboveGround_10_10u,ldaps_g11_heightAboveGround_10_10u,ldaps_g12_heightAboveGround_10_10u,ldaps_g13_heightAboveGround_10_10u,ldaps_g14_heightAboveGround_10_10u,ldaps_g15_heightAboveGround_10_10u,ldaps_g16_heightAboveGround_10_10u,ldaps_g1_heightAboveGround_10_10v,ldaps_g2_heightAboveGround_10_10v,ldaps_g3_heightAboveGround_10_10v,ldaps_g4_heightAboveGround_10_10v,ldaps_g5_heightAboveGround_10_10v,ldaps_g6_heightAboveGround_10_10v,ldaps_g7_heightAboveGround_10_10v,ldaps_g8_heightAboveGround_10_10v,...,ldaps_g9_surface_0_lsm,ldaps_g10_surface_0_lsm,ldaps_g11_surface_0_lsm,ldaps_g12_surface_0_lsm,ldaps_g13_surface_0_lsm,ldaps_g14_surface_0_lsm,ldaps_g15_surface_0_lsm,ldaps_g16_surface_0_lsm,ldaps_g1_surface_0_h,ldaps_g2_surface_0_h,ldaps_g3_surface_0_h,ldaps_g4_surface_0_h,ldaps_g5_surface_0_h,ldaps_g6_surface_0_h,ldaps_g7_surface_0_h,ldaps_g8_surface_0_h,ldaps_g9_surface_0_h,ldaps_g10_surface_0_h,ldaps_g11_surface_0_h,ldaps_g12_surface_0_h,ldaps_g13_surface_0_h,ldaps_g14_surface_0_h,ldaps_g15_surface_0_h,ldaps_g16_surface_0_h,data_available_kst_dtm
0,2022-01-01 01:00:00,5.411171,5.593300,4.383828,3.022011,4.837441,6.17875,6.027382,5.132363,3.042031,4.557656,6.052285,6.609902,6.231972,4.974648,6.134804,6.571816,1.591679,0.727909,-0.924435,0.641972,0.774784,0.774784,0.015507,-1.101193,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,992.625,936.5625,868.8125,926.5,997.625,1001.25,934.4375,869.4375,889.1875,966.6875,999.0625,959.75,896.25,956.625,967.875,933.75,2021-12-31 13:00:00
1,2022-01-01 02:00:00,4.968168,4.550687,3.019926,1.975980,4.179105,5.13516,4.400297,3.527738,2.328031,4.047269,5.318754,5.435941,4.739652,4.507230,5.579496,5.823148,0.875631,-0.232767,-1.829935,0.937643,0.703268,0.314596,-0.834330,-1.945169,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,992.625,936.5625,868.8125,926.5,997.625,1001.25,934.4375,869.4375,889.1875,966.6875,999.0625,959.75,896.25,956.625,967.875,933.75,2021-12-31 13:00:00


**확인할 것**
- `ldaps_train.shape[0]`이 26304, `ldaps_test.shape[0]`이 8760과 같은지
- `ldaps_train_raw["grid_id"].nunique()`가 16인지
- 컬럼 수가 `(16개 격자 x 변수 수) + 2`와 맞는지

### 1-3. data_available_kst_dtm 규칙 검증

data_description.md는 "01:00부터 다음날 00:00까지의 24개 예보 대상 시각은 같은 data_available_kst_dtm을 갖는다"고 설명합니다. 이건 예측기준시점 원칙(leakage-guard)의 전제가 되는 규칙이라, 실제 데이터로 맞는지 검증해야 나중에 피처를 만들 때 안심하고 쓸 수 있습니다.

In [8]:
for name, df in [("gfs_train", gfs_train), ("ldaps_train", ldaps_train)]:
    # 블록은 "이날 01:00 ~ 다음날 00:00"의 24시간 단위이므로, 1시간을 당기면 달력 날짜와 블록이 정확히 겹칩니다.
    block_date = (df["forecast_kst_dtm"] - pd.Timedelta(hours=1)).dt.date
    n_unique_per_block = df.groupby(block_date)["data_available_kst_dtm"].nunique()
    print(name, "- 블록(01:00~익일 00:00) 안에서 data_available_kst_dtm이 2개 이상인 블록 수:", (n_unique_per_block > 1).sum(), "/", len(n_unique_per_block))

gfs_train - 블록(01:00~익일 00:00) 안에서 data_available_kst_dtm이 2개 이상인 블록 수: 0 / 1096
ldaps_train - 블록(01:00~익일 00:00) 안에서 data_available_kst_dtm이 2개 이상인 블록 수: 0 / 1096


**확인할 것**: 두 결과 모두 0이어야 합니다. (처음 실행 결과가 1095였다면 제 검증 코드의 버그였습니다 — 예보 블록이 "이날 01:00~다음날 00:00"으로 자정을 넘어가는데, 제가 처음엔 달력 날짜로만 묶어서 자정(00:00) 값이 다음 날짜 그룹에 섞여 들어갔었습니다. 1시간을 당겨서 블록 기준으로 다시 묶으니 위반 0건으로 확인됐습니다 — 실제 데이터에는 문제가 없었고 제 체크 로직이 문제였습니다.)

## 2. SCADA — 10분 단위 → 1시간 단위, 터빈 → KPX 그룹 합산

`data/info.xlsx`를 열어보면 터빈별 KPX 그룹 소속이 나와 있습니다 (아래 셀에서 그대로 출력해서 눈으로 확인합니다).

In [9]:
info = pd.read_excel(DATA_DIR / "info.xlsx", sheet_name="info", header=3)
info = info.dropna(subset=["호기"]).reset_index(drop=True)
info["KPX그룹"] = info["KPX그룹"].ffill()  # 병합 셀이라 그룹 시작 행에만 값이 있고 나머지는 빈칸
info[["단계", "제작사", "호기", "KPX그룹", "설비용량(MW)"]]

,단계,제작사,호기,KPX그룹,설비용량(MW)
0,1,VESTAS,1,1.0,3.6
1,1,VESTAS,2,1.0,3.6
2,1,VESTAS,3,1.0,3.6
3,1,VESTAS,4,1.0,3.6
4,1,VESTAS,5,1.0,3.6
5,1,VESTAS,6,1.0,3.6
6,1,VESTAS,7,2.0,3.6
7,1,VESTAS,8,2.0,3.6
8,1,VESTAS,9,2.0,3.6
9,1,VESTAS,10,2.0,3.6


**확인할 것**: VESTAS 1~6호기가 그룹 1, VESTAS 7~12호기가 그룹 2, UNISON 1~5호기가 그룹 3으로 나오는지. (그룹설비용량 21.6/21.6/21 MW과도 맞아야 합니다.)

In [10]:
# 위 표에서 확인한 매핑을 딕셔너리로 고정 (터빈 구성은 바뀌지 않는 고정 메타정보이므로 하드코딩)
TURBINE_GROUP_MAP = {}
for i in range(1, 7):
    TURBINE_GROUP_MAP[f"vestas_wtg{i:02d}"] = "kpx_group_1"
for i in range(7, 13):
    TURBINE_GROUP_MAP[f"vestas_wtg{i:02d}"] = "kpx_group_2"
for i in range(1, 6):
    TURBINE_GROUP_MAP[f"unison_wtg{i:02d}"] = "kpx_group_3"

TURBINE_GROUP_MAP

{'vestas_wtg01': 'kpx_group_1',
 'vestas_wtg02': 'kpx_group_1',
 'vestas_wtg03': 'kpx_group_1',
 'vestas_wtg04': 'kpx_group_1',
 'vestas_wtg05': 'kpx_group_1',
 'vestas_wtg06': 'kpx_group_1',
 'vestas_wtg07': 'kpx_group_2',
 'vestas_wtg08': 'kpx_group_2',
 'vestas_wtg09': 'kpx_group_2',
 'vestas_wtg10': 'kpx_group_2',
 'vestas_wtg11': 'kpx_group_2',
 'vestas_wtg12': 'kpx_group_2',
 'unison_wtg01': 'kpx_group_3',
 'unison_wtg02': 'kpx_group_3',
 'unison_wtg03': 'kpx_group_3',
 'unison_wtg04': 'kpx_group_3',
 'unison_wtg05': 'kpx_group_3'}

### 2-1. 리샘플링 방법 (근거 — 검증으로 확정)

처음엔 "`_power_kw10m`은 10분간 평균출력(kW)이니, 1시간 발전량(kWh) = 6개 값의 평균"이라고 가정했습니다. 그런데 실제로 계산해서 `train_labels`와 비교해보니 **상관계수가 거의 0**이 나왔습니다. 원인을 파고들어 아래 두 가지를 찾았습니다.

**1) 먼저 물리적으로 불가능한 값(센서 오류) 제거**: VESTAS 데이터에 절댓값이 수천만에 달하는 값(터빈 1기 설비용량은 3,600kW인데 5천만kW 등)이 섞여 있었습니다. 통신 오류로 생긴 튀는 값으로 판단해 `abs(power) > 10,000`인 값은 결측치(NaN) 처리합니다. UNISON에는 이런 값이 없었습니다.

**2) 집계 방법을 평균이 아니라 합계로 검증**: 오류값을 제거한 뒤 '평균'과 '합계' 두 방식을 모두 계산해서 `train_labels`와 상관·비율을 비교했습니다.
- 평균 방식: 상관 거의 0
- **합계 방식(6개 10분값을 그대로 더함)**: 상관 0.9998(그룹1)/0.9998(그룹2)/0.9966(그룹3), label 대비 비율도 0.99~1.01로 거의 일치

즉 `_power_kw10m` 컬럼명은 "10분 평균출력"처럼 보이지만, **실제로는 이미 "10분간 발전량(kWh)"에 해당하는 값**이라는 뜻입니다(컬럼명과 실제 단위가 다를 수 있다는 걸 데이터로 직접 검증한 사례입니다 — 문서보다 데이터가 우선). 따라서 1시간 발전량 = 6개 값을 그냥 더한 값입니다.

**데이터 사이언티스트 관점**: 가정(평균)을 바로 코드에 반영하지 않고, 실제 라벨과 대조해서 상관·비율로 검증했기 때문에 이 결론은 "그럴듯해서"가 아니라 근거(실험 결과)로 확정된 것입니다.
**도메인 전문가 관점**: SCADA 로거가 관례적으로 "kW" 단위 이름을 쓰면서 실제로는 10분 적산 에너지(kWh)를 기록하는 경우가 실제로 흔합니다. 극단치도 진짜 발전량이 아니라 통신 오류로 보는 게 물리적으로 타당합니다(3.6MW 터빈이 순간적으로 5천만 kW를 낼 수는 없음).

시간 경계도 맞춰야 합니다. `train_labels.csv`의 `kst_dtm`은 "집계 구간의 종료 시각"이므로(예: `01:00:00` = 00:10~01:00 구간), `resample("h", closed="right", label="right")`로 오른쪽 끝 시각을 라벨로 씁니다.

In [11]:
scada_vestas_raw = pd.read_csv(DATA_DIR / "train" / "scada_vestas_train.csv", encoding="utf-8-sig", parse_dates=["kst_dtm"])
scada_unison_raw = pd.read_csv(DATA_DIR / "train" / "scada_unison_train.csv", encoding="utf-8-sig", parse_dates=["kst_dtm"])

print(scada_vestas_raw.shape, scada_unison_raw.shape)
print(scada_vestas_raw["kst_dtm"].min(), "~", scada_vestas_raw["kst_dtm"].max())

(157819, 37) (105264, 16)
2022-01-01 01:00:00 ~ 2025-01-01 00:00:00


In [12]:
EXTREME_POWER_THRESHOLD = 10_000  # 터빈 1기 최대 설비용량(4,200kW)보다 훨씬 큰, 물리적으로 불가능한 값 기준


def hourly_group_power(df: pd.DataFrame) -> pd.DataFrame:
    """10분 단위 터빈별 power_kw10m -> 1시간 단위 KPX 그룹별 발전량(kWh) 합계."""
    power_cols = [c for c in df.columns if c.endswith("_power_kw10m")]

    clean = df.copy()
    n_flagged = 0
    for c in power_cols:
        bad = clean[c].abs() > EXTREME_POWER_THRESHOLD
        n_flagged += int(bad.sum())
        clean.loc[bad, c] = np.nan
    print(f"  - 물리적으로 불가능한 값(|power|>{EXTREME_POWER_THRESHOLD}) 결측 처리: {n_flagged}건")

    # _power_kw10m은 검증 결과 '10분간 발전량(kWh)'이므로, 1시간 발전량 = 6개 값의 합계
    hourly = clean.set_index("kst_dtm")[power_cols].resample("h", closed="right", label="right").sum(min_count=1)

    group_result = {}
    for col in power_cols:
        turbine = col.replace("_power_kw10m", "")
        group = TURBINE_GROUP_MAP[turbine]
        group_result.setdefault(group, []).append(col)

    return pd.DataFrame({group: hourly[cols].sum(axis=1, min_count=1) for group, cols in group_result.items()})


scada_hourly = pd.concat([hourly_group_power(scada_vestas_raw), hourly_group_power(scada_unison_raw)], axis=1)
scada_hourly = scada_hourly.add_prefix("scada_").reset_index()

print(scada_hourly.shape)
scada_hourly.head(3)

  - 물리적으로 불가능한 값(|power|>10000) 결측 처리: 868건
  - 물리적으로 불가능한 값(|power|>10000) 결측 처리: 0건
(26304, 4)


,kst_dtm,scada_kpx_group_1,scada_kpx_group_2,scada_kpx_group_3
0,2022-01-01 01:00:00,2005.0,1536.0,NaN
1,2022-01-01 02:00:00,13090.0,10410.0,NaN
2,2022-01-01 03:00:00,12188.0,10880.0,NaN


**확인할 것**: 행 수가 SCADA 기간(대략 2022~2024, VESTAS/UNISON 시작 시점이 다를 수 있음)의 시간 수와 비슷한지, 컬럼이 `scada_kpx_group_1/2/3` 3개인지.

### 2-2. 정합성 확인 — SCADA 합계 vs train_labels

터빈-그룹 매핑과 리샘플링(시간 경계 포함)이 맞았다면, SCADA를 합산한 값이 실제 KPX 라벨과 아주 높은 상관을 보여야 합니다. 여기서 상관이 낮게 나오면 매핑/시간 정렬이 잘못됐다는 신호이니, 다음 단계(조인)로 넘어가기 전에 반드시 확인합니다.

In [13]:
labels = pd.read_csv(DATA_DIR / "train" / "train_labels.csv", encoding="utf-8-sig", parse_dates=["kst_dtm"])
check = labels.merge(scada_hourly, on="kst_dtm", how="inner")

for col in GROUP_COLS:
    scada_col = f"scada_{col}"
    sub = check[[col, scada_col]].dropna()
    corr = sub[col].corr(sub[scada_col])
    print(col, "- 겹치는 시간 수:", len(sub), "| 라벨-SCADA 상관:", round(corr, 4))

kpx_group_1 - 겹치는 시간 수: 26200 | 라벨-SCADA 상관: 0.9998
kpx_group_2 - 겹치는 시간 수: 26201 | 라벨-SCADA 상관: 0.9998
kpx_group_3 - 겹치는 시간 수: 17534 | 라벨-SCADA 상관: 0.9966


**확인할 것**: 상관이 세 그룹 모두 약 0.9998 / 0.9998 / 0.9966 근처로 나와야 정상입니다 (완전히 1이 아닌 이유: SCADA는 터빈 단, 라벨은 KPX 정산 기준이라 계통 손실 등 약간의 차이가 있을 수 있음). 이 값과 크게 다르게 나오면 바로 알려주세요 — 그 경우 셀 실행 환경 차이(pandas 버전 등)를 같이 확인해야 합니다.

## 3. 최종 조인 — train_base / test_base

- **train_base**: `kst_dtm` 기준으로 라벨 + GFS + LDAPS + SCADA를 모두 조인 (SCADA는 분석·피처 아이디어 검증용으로만 남겨두고, 실제 test에는 없다는 점을 항상 기억)
- **test_base**: 라벨도 SCADA도 없이 GFS + LDAPS만 조인 (실제로 2025년엔 이 두 가지만 존재)

In [14]:
gfs_train_r = gfs_train.rename(columns={"forecast_kst_dtm": "kst_dtm"})
ldaps_train_r = ldaps_train.rename(columns={"forecast_kst_dtm": "kst_dtm", "data_available_kst_dtm": "data_available_kst_dtm_ldaps"})
gfs_train_r = gfs_train_r.rename(columns={"data_available_kst_dtm": "data_available_kst_dtm_gfs"})

train_base = labels.merge(gfs_train_r, on="kst_dtm", how="left")
train_base = train_base.merge(ldaps_train_r, on="kst_dtm", how="left")
train_base = train_base.merge(scada_hourly, on="kst_dtm", how="left")

print(train_base.shape)
print("결측 총량:", train_base.isna().sum().sum())

(26304, 804)
결측 총량: 17741


In [15]:
gfs_test_r = gfs_test.rename(columns={"forecast_kst_dtm": "kst_dtm", "data_available_kst_dtm": "data_available_kst_dtm_gfs"})
ldaps_test_r = ldaps_test.rename(columns={"forecast_kst_dtm": "kst_dtm", "data_available_kst_dtm": "data_available_kst_dtm_ldaps"})

test_base = gfs_test_r.merge(ldaps_test_r, on="kst_dtm", how="left")

print(test_base.shape)
print("결측 총량:", test_base.isna().sum().sum())

(8760, 798)
결측 총량: 752


**확인할 것**
- `train_base.shape[0]`이 26304와 같은지 (라벨 기준으로 왼쪽 조인했으므로 행 수는 유지되어야 함)
- `test_base.shape[0]`이 8760과 같은지
- GFS/LDAPS 결측이 0에 가까운지 (있다면 예보 누락일 후보 — 몇 건인지 원인 파악 필요)
- `scada_kpx_group_*` 컬럼은 SCADA 시작 이전 구간에서 결측이 있는 게 정상 (SCADA가 2023년부터라서)

## 4. 저장 (parquet 캐시)

In [16]:
train_base.to_parquet(PROCESSED_DIR / "train_base.parquet", index=False)
test_base.to_parquet(PROCESSED_DIR / "test_base.parquet", index=False)
gfs_grid_meta.to_parquet(PROCESSED_DIR / "gfs_grid_meta.parquet", index=False)
ldaps_grid_meta.to_parquet(PROCESSED_DIR / "ldaps_grid_meta.parquet", index=False)

print("저장 완료:", [p.name for p in PROCESSED_DIR.glob("*.parquet")])

저장 완료: ['gfs_grid_meta.parquet', 'ldaps_grid_meta.parquet', 'test_base.parquet', 'train_base.parquet']


## 요약 및 다음 단계

이 노트북은 구조 변환만 했습니다 (pivot, 리샘플링, 조인) — 새로운 피처는 아직 없습니다.

**실행 방법**: 위에서부터 순서대로 실행하고, 각 '확인할 것' 결과(특히 shape 숫자, 상관계수, 결측 총량)를 알려주세요. 특히 2-2 SCADA-라벨 상관은 꼭 확인 부탁드립니다.

확인되면:
1. `reports/01_preprocessing.md` 작성
2. `02_eda.ipynb`의 2~4절(기상 예보 격자 지도, SCADA 파워커브 등)을 이 parquet 캐시를 불러와서 이어가겠습니다.